## 2-1: 코마 구분 리스트 파서 활용

In [3]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출comma력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [4]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [8]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import csv

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()

# 출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()


# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

# LLM 설정 (제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
llm = get_llm(temperature=0.7)


[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [7]:

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | llm | output_parser

# "AI 관련 기술"에 대한 체인 호출 실행
result = chain.invoke({"subject": "음식"})

# 쉼표로 구분된 리스트 출력
print("추천 음식 리스트:")
print(result)

추천 음식 리스트:
['김치', '불고기', '비빔밥', '잡채', '떡볶이']


## 2-2: 영화 리뷰 감정 분석기

In [11]:
from langchain_classic.output_parsers import OutputFixingParser
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate


# 허용할 분류 값을 Enum 으로 정의 (이름 = 코드에서 쓰는 값, 값 = 모델이 답해야 하는 문자열)
class Sentiment(Enum):
    """리뷰 감성 분류 값"""
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "중립"


# Enum 출력 파서 초기화
parser = EnumOutputParser(enum=Sentiment)

# FixingParser 사용 => 잘못된 판단을 바로잡아줌
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

# 출력 형식 지침 확인: "Select one of the following options: 긍정, 부정, 중립"
format_instructions = fixing_parser.get_format_instructions()
print(format_instructions)

# 모델 없이 파서만 테스트 (앞뒤 공백/줄바꿈은 제거하고 Enum 으로 변환)
print(repr(parser.parse("긍정")))
print(repr(parser.parse(" 부정\n")))

# 프롬프트 템플릿 설정 (형식 지침을 미리 주입)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 리뷰 감성 분류 전문가입니다. 리뷰를 읽고 감성을 분류하세요. 설명 없이 값만 답하세요.\n{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)

# LLM 초기화 (분류 작업이므로 temperature 를 0으로 설정)
llm = get_llm(temperature=0)

# 체인 생성: 프롬프트 → 모델 → Enum 파서
chain = prompt | llm | parser

Select one of the following options: 긍정, 부정, 중립
<Sentiment.POSITIVE: '긍정'>
<Sentiment.NEGATIVE: '부정'>
[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [12]:
from langchain_core.exceptions import OutputParserException

reviews = [
    # 기대 정답: 긍정
    "처음부터 끝까지 한순간도 지루하지 않았어요. 배우들 연기가 정말 자연스럽고, "
    "마지막 장면에서는 눈물이 날 정도로 감동적이었습니다. 친구들한테 꼭 보라고 추천할 거예요!",

    # 기대 정답: 부정
    "예고편만 보고 기대했는데 완전 실망이었어요. 이야기가 너무 뻔하고 중간부터는 지루해서 "
    "졸 뻔했습니다. 두 시간이 아까웠어요.",

    # 기대 정답: 중립
    "영상미와 음악은 괜찮았지만 스토리는 평범했어요. 크게 감동적이지도, 그렇다고 "
    "아주 별로이지도 않은 무난한 영화였습니다. 시간 날 때 한 번쯤 볼 만해요.",
]

for review in reviews:
    print(f"리뷰: {review}")
    try:
        result = chain.invoke({"review": review})
        # 결과는 문자열이 아니라 Sentiment Enum 멤버
        print(f"  분류 결과: {result} / 타입: {type(result).__name__} / 값: {result.value}")
    except OutputParserException as e:
        # 모델이 Enum 에 없는 값으로 답하면 파싱 오류가 발생
        print(f"  파싱 오류: {e}")
    print("-" * 50)

리뷰: 처음부터 끝까지 한순간도 지루하지 않았어요. 배우들 연기가 정말 자연스럽고, 마지막 장면에서는 눈물이 날 정도로 감동적이었습니다. 친구들한테 꼭 보라고 추천할 거예요!
  분류 결과: Sentiment.POSITIVE / 타입: Sentiment / 값: 긍정
--------------------------------------------------
리뷰: 예고편만 보고 기대했는데 완전 실망이었어요. 이야기가 너무 뻔하고 중간부터는 지루해서 졸 뻔했습니다. 두 시간이 아까웠어요.
  분류 결과: Sentiment.NEGATIVE / 타입: Sentiment / 값: 부정
--------------------------------------------------
리뷰: 영상미와 음악은 괜찮았지만 스토리는 평범했어요. 크게 감동적이지도, 그렇다고 아주 별로이지도 않은 무난한 영화였습니다. 시간 날 때 한 번쯤 볼 만해요.
  분류 결과: Sentiment.NEUTRAL / 타입: Sentiment / 값: 중립
--------------------------------------------------


## 학생 정보 구조화 시스템

In [16]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# Pydantic 모델 정의 (테이블 데이터 구조화)
class TableData(BaseModel):
    name: str = Field(..., description="이름")
    age: int = Field(..., description="나이 (숫자)")
    major: str = Field(..., description="전공 학과")
    hobbies: list[str] = Field(..., description="취미 목록")
    goal: str = Field(..., description="목표")

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=TableData)
# print("출력지침 = " + parser.)

# ChatPromptTemplate 설정 (최신 LCEL 스타일)
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 테이블 형태의 데이터를 생성하는 AI 어시스턴트입니다.
    매번 **동일한 데이터**를 정확히 동일한 순서로 반환하세요.
    반드시 다음 JSON 스키마를 정확히 따라야 합니다:
    {format_instructions}
    
    데이터는 현실적이고 일관성 있게 생성하세요. (실제 통계가 아닌 예시용 가상 데이터입니다.)"""),
    ("user", "{query}")
])

# 부분 변수 설정
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 체인 생성
chain = prompt | llm | parser


In [18]:
import json

# 질문 설정 
query = "안녕하세요! 저는 박원빈이고 22살입니다. 실용음악과를 전공하고 있어요. 취미로는 게임하기, 영화보기, 집에서 뒹굴기를 좋아합니다. 훌륭한 아티스트가 되는 것이 목표입니다. \
이름은 `name`에, 나이는 `age`에, 학과는 `major`에, 취미는 `hobbies`에 배열로 담아주고,  목표는 `goal`에 담아 주세요."

response = chain.invoke({"query": query})
print(json.dumps(response.model_dump(), indent=4, ensure_ascii=False))

{
    "name": "박원빈",
    "age": 22,
    "major": "실용음악과",
    "hobbies": [
        "게임하기",
        "영화보기",
        "집에서 뒹굴기"
    ],
    "goal": "훌륭한 아티스트가 되는 것이 목표입니다."
}


## 2-4: 여행 계획 분석기

In [21]:
import json
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate

# 출력할 필드를 ResponseSchema 로 정의 (name: 결과 dict 의 키, description: 모델에게 알려줄 설명)
response_schemas = [
    ResponseSchema(
        name="destination",
        description="여행지 (도시 또는 국가 이름). 예: 제주도, 도쿄, 프랑스",
    ),
    ResponseSchema(
        name="duration",
        description="여행 기간. 예: 3박 4일, 일주일. 텍스트에 없으면 null",
    ),
    ResponseSchema(
        name="budget",
        description="여행 예산. 예: 100만 원. 텍스트에 없으면 null",
    ),
    ResponseSchema(
        name="rating",
        description="원하는 숙소 등급. 예: 3성급, 5성급. 텍스트에 없으면 null",
    ),
    ResponseSchema(
        name="activities",
        description="하고 싶은 활동 목록. 예: [\"스노클링\", \"맛집 탐방\"]. 없으면 빈 배열 []",
        type="array",   # 활동은 여러 개일 수 있으니까 배열로!
    ),
]

# ResponseSchema 목록으로 파서 생성
parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 출력 형식 지침 확인: ```json 코드 블록과 필드별 설명이 포함됨
format_instructions = parser.get_format_instructions()

# ① 프롬프트 만들기
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 사용자의 여행 계획 글에서 정보를 정확히 추출하는 AI 어시스턴트입니다.\n"
     "텍스트에 없는 정보는 지어내지 말고 null로 두세요.\n"
     "반드시 다음 형식을 따르세요:\n{format_instructions}"),
    ("user", "{query}"),
])

# ② 빈칸 미리 채우기
prompt = prompt.partial(format_instructions=format_instructions)

# LLM 초기화 (구조화 출력은 일관성이 중요하므로 temperature 를 0으로 설정)
llm = get_llm(temperature=0)

# 체인 생성: 프롬프트 → 모델 → 구조화 파서
chain = prompt | llm | parser


[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [ ]:
import json

# 질문 설정
query = (
    "다음 달에 친구랑 제주도로 3박 4일 여행을 가려고 해요. "
    "예산은 1인당 80만 원 정도 생각하고 있고, 숙소는 4성급 호텔이면 좋겠어요. "
    "가서 스노클링도 하고, 한라산 등산이랑 흑돼지 맛집 탐방도 하고 싶어요!"
)

response = chain.invoke({"query": query})
print(json.dumps(response, indent=4, ensure_ascii=False))


{
    "destination": "제주도",
    "duration": "3박 4일",
    "budget": "1인당 80만 원",
    "rating": "4성급 호텔",
    "activities": [
        "스노클링",
        "한라산 등산",
        "흑돼지 맛집 탐방"
    ]
}
